# Notebook 027: Aggregate & Cumulative Window Functions: Running Totals and Moving Averages

**Module:** Module 03: Advanced SQL, Aggregations & Analytical Window Functions  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 20: Finance Domain E2E Project & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A fintech wealth management app displays account balance trajectories and 7-day smoothed moving averages of daily expenditures to help customers track spending habits.

## 2. Technical Requirement & Problem Statement
Implement cumulative sums (running totals), moving averages, and frame boundary specifications: compare physical row-based frames (`rowsBetween`) vs logical value-based frames (`rangeBetween`). Calculate moving averages over preceding windows.

## 3. Spark & Databricks Architecture Notes
The default window frame for an ordered window is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. This can cause subtle bugs when duplicate order keys exist, as Spark aggregates all rows with the identical key together. Using `ROWS BETWEEN` guarantees explicit physical row framing.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_027_Aggregate_&_Cumulative_Window_Functions_Running_Totals_and_Moving_Averages")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Daily Spending Feed
Create daily customer transactions.

In [ ]:
spending = [
    ("ACC-1", "2023-11-01", 120.0),
    ("ACC-1", "2023-11-02", 45.0),
    ("ACC-1", "2023-11-03", 210.0),
    ("ACC-1", "2023-11-04", 80.0),
    ("ACC-1", "2023-11-05", 15.0),
    ("ACC-1", "2023-11-06", 350.0),
    ("ACC-1", "2023-11-07", 90.0)
]

schema = "account_id STRING, txn_date STRING, amount DOUBLE"
df_spend = spark.createDataFrame(spending, schema)
df_spend.show()

## 6. Cumulative Running Total & 3-Day Moving Average
Apply rowsBetween frames for physical sliding windows.

In [ ]:
# Cumulative window from start to current row
cum_window = (
    Window.partitionBy("account_id")
    .orderBy("txn_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# 3-day moving window: 2 preceding rows plus current row
moving_window = (
    Window.partitionBy("account_id")
    .orderBy("txn_date")
    .rowsBetween(-2, Window.currentRow)
)

df_analytics = df_spend.withColumn(
    "cumulative_spend", F.round(F.sum("amount").over(cum_window), 2)
).withColumn(
    "moving_avg_3day", F.round(F.avg("amount").over(moving_window), 2)
)

print("Cumulative Spending & Moving Average Analysis:")
df_analytics.show()

## 7. Comparing ROWS BETWEEN vs RANGE BETWEEN with Duplicate Keys
Illustrate how duplicate keys affect RANGE frames vs ROWS frames.

In [ ]:
dup_data = [
    ("A", 1, 10.0),
    ("A", 2, 20.0),
    ("A", 2, 30.0),  # Duplicate order key
    ("A", 3, 40.0)
]
df_dup = spark.createDataFrame(dup_data, ["id", "val", "amt"])

# ROWS vs RANGE
w_rows = Window.partitionBy("id").orderBy("val").rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_range = Window.partitionBy("id").orderBy("val").rangeBetween(Window.unboundedPreceding, Window.currentRow)

df_comp = df_dup.withColumn("sum_rows", F.sum("amt").over(w_rows))                .withColumn("sum_range", F.sum("amt").over(w_range))

print("Notice sum_range evaluates tied val=2 together into 60.0, while sum_rows evaluates row-by-row:")
df_comp.show()

## 9. Key Architectural Takeaways
- **`ROWS BETWEEN` defines physical row count boundaries (e.g. 2 preceding rows).**
- **`RANGE BETWEEN` defines logical value offsets based on the order column.**
- **Default window frame with `orderBy` is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, which aggregates identical order keys together.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between `rowsBetween` and `rangeBetween` in Spark Window specifications?
2. What is the default frame specification when an `orderBy` clause is present in a Window?
3. How do you compute a rolling 30-day time-based window on irregular transaction dates?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
